# ⚙️ Урок 19 — AutoML (материалы преподавателя)

> 🎯 Цель: показать перебор моделей (AutoML), сравнить с ручной моделью, взять модель с Hugging Face Hub.

Блок 1 работает офлайн. Блоки 2 и 4 — в Colab с интернетом.

## Блок 1 · Мини-AutoML: перебор моделей (офлайн)
**Идея:** сами перебираем несколько моделей и строим лидерборд — ровно то, что AutoML делает автоматически.

In [ ]:
import seaborn as sns, warnings; warnings.filterwarnings('ignore')
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

df = sns.load_dataset('titanic')
num=['age','fare','sibsp','parch']; cat=['sex','pclass','embarked']
X=df[num+cat]; y=df['survived']
prep=ColumnTransformer([
    ('n',Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]),num),
    ('c',Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]),cat)])

models={'KNN':KNeighborsClassifier(),'Дерево':DecisionTreeClassifier(random_state=42),
        'ЛогРег':LogisticRegression(max_iter=1000),'Лес':RandomForestClassifier(random_state=42),
        'Бустинг':GradientBoostingClassifier(random_state=42)}
board=[]
for name,m in models.items():
    s=cross_val_score(Pipeline([('prep',prep),('model',m)]),X,y,cv=5).mean()
    board.append((name,s))
print('=== Лидерборд (мини-AutoML) ===')
for name,s in sorted(board,key=lambda x:-x[1]):
    print(f'{name:10s} {s:.1%}')

## Блок 2 (в Colab) · Настоящий AutoGluon одной командой

In [ ]:
!pip install autogluon.tabular -q
from autogluon.tabular import TabularPredictor
import seaborn as sns
df = sns.load_dataset('titanic')[['survived','sex','pclass','age','fare','sibsp','parch','embarked']]
train=df.sample(frac=0.8,random_state=42); test=df.drop(train.index)
predictor=TabularPredictor(label='survived').fit(train, time_limit=120)
predictor.leaderboard(test)

## Блок 3 · Сравнение с ручной моделью (урок 10–12)
Наша ручная модель давала ~81%. Сравни с верхней строкой лидерборда.

In [ ]:
print('Ручная модель (урок 10-12): ~81%')
print('AutoML лучшая: см. лидерборд')
# Часто AutoML немного выигрывает — за счёт перебора и ансамблей.

## Блок 4 (в Colab) · Готовая модель с Hugging Face Hub

In [ ]:
!pip install transformers -q
from transformers import pipeline
translator = pipeline('translation', model='Helsinki-NLP/opus-mt-en-ru')
print(translator('Machine learning is fun')[0]['translation_text'])

---
**Итог.** AutoML перебирает модели и выдаёт лучшую, но не отменяет проверку (утечка, метрика). Hub — склад готовых моделей: часто обучать не нужно.